# ACE reproduction on Formula (FTEC5660)
Reproduces Table 2 of *Agentic Context Engineering* (ICLR 2026) with the DeepSeek API instead of DeepSeek-V3.1 on SambaNova.
Put the key in Colab Secrets as `DEEPSEEK_API_KEY`, or type it when asked. A CPU runtime is enough.
The setup cell clones the fork, or resets an existing checkout to it, so it runs exactly what is on GitHub.


In [ ]:
import getpass, pathlib, subprocess
%cd /content
# update in place: results/ and .env are git-ignored, so earlier runs survive a re-run of this cell
!if [ -d ace/.git ]; then git -C ace fetch -q && git -C ace reset -q --hard origin/main; else git clone -q https://github.com/baimingyang98/ace.git; fi
%cd /content/ace
!git log --oneline -1
!pip -q install "openai>=1.0.0" python-dotenv tiktoken

key = None
try:
    from google.colab import userdata
    key = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    pass
if not key:
    key = getpass.getpass("DEEPSEEK_API_KEY: ")
pathlib.Path(".env").write_text(f"DEEPSEEK_API_KEY={key}\n")
print("key written to .env (not printed)")

MODEL = "deepseek-v4-pro"   # or "deepseek-flash"
MAX_ROUNDS = 5              # paper section 4.2 uses 5 Reflector rounds; the repo default is 3
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()


In [ ]:
import glob, json, os, re, time
from importlib.metadata import version

# USD per 1M tokens, off-peak, from api-docs.deepseek.com/quick_start/pricing (read 2026-10-24); peak hours cost 2x
PRICES = {"deepseek-flash": (0.15, 0.6), "deepseek-v4-pro": (0.66, 1.98)}


def run_ace(save_path, mode="online", task="formula", model=MODEL, rounds=MAX_ROUNDS, extra=(), env=None, cwd="."):
    """Run eval/finance/run.py with the output in a log file; skip if this save_path already finished."""
    done = glob.glob(os.path.join(cwd, save_path, "ace_run_*", "final_results.json"))
    if done:
        print("already done:", done[0])
        return
    os.makedirs(os.path.join(cwd, save_path), exist_ok=True)
    cmd = ["python", "-m", "eval.finance.run", "--task_name", task, "--mode", mode, "--api_provider", "deepseek",
           "--generator_model", model, "--reflector_model", model, "--curator_model", model,
           "--max_num_rounds", str(rounds), "--save_path", save_path, *extra]
    log_path = os.path.join(cwd, save_path, "stdout.log")
    if os.path.exists(log_path):  # keep the log of an earlier, unfinished attempt
        os.rename(log_path, log_path.replace(".log", f"_{int(os.path.getmtime(log_path))}.log"))
    t0 = time.time()
    with open(log_path, "w") as log:
        out = subprocess.run(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
    print(f"exit {out.returncode}, {(time.time() - t0) / 60:.1f} min")
    text = open(os.path.join(cwd, save_path, "stdout.log")).read()
    print(text[-1500:])
    if out.returncode != 0:
        # stop here: a failed run (e.g. HTTP 402, out of balance) would make every later run fail the same way
        errors = re.findall(r"Error code: \d+.*", text)
        raise RuntimeError(f"run failed: {errors[-1][:300] if errors else 'see stdout.log'}")


def summarize(save_path, model=MODEL, cwd="."):
    """Accuracy, calls, tokens, cost and silent failures of the latest run under save_path.
    A run that crashed has no final_results.json; then the base accuracy comes from initial_test_results.json
    and the ACE accuracy is the cumulative online accuracy at the last finished window."""
    run_dir = sorted(glob.glob(os.path.join(cwd, save_path, "ace_run_*")))[-1]
    final = os.path.join(run_dir, "final_results.json")
    if os.path.exists(final):
        res = json.load(open(final))
    else:
        res = {}
        initial = os.path.join(run_dir, "initial_test_results.json")
        if os.path.exists(initial):
            res["initial_test_results"] = json.load(open(initial))["test_results"]
        windows = re.findall(r"Cumulative test accuracy so far: ([\d.]+) \((\d+) samples\)",
                             open(os.path.join(cwd, save_path, "stdout.log")).read())
        if windows:
            res["online_test_results"] = {"accuracy": float(windows[-1][0]), "samples": int(windows[-1][1])}
    calls = [json.load(open(p)) for p in glob.glob(os.path.join(run_dir, "detailed_llm_logs", "*.json"))]
    ok = [c for c in calls if "prompt_num_tokens" in c]
    tin = sum(c["prompt_num_tokens"] for c in ok)
    tout = sum(c["response_num_tokens"] for c in ok)
    pin, pout = PRICES[model]
    by_role = {r: sum(c["role"] == r for c in calls) for r in ("generator", "reflector", "curator")}
    row = {
        "run": save_path,
        "complete": os.path.exists(final),
        "online_samples_scored": res.get("online_test_results", {}).get("samples",
                                                                         res.get("online_test_results", {}).get("total")),
        "base_acc": res.get("initial_test_results", {}).get("accuracy", res.get("test_results", {}).get("accuracy")),
        "ace_acc": res.get("online_test_results", {}).get("accuracy",
                                                          res.get("final_test_results", {}).get("accuracy")),
        "calls": len(calls), **by_role,
        "failed_or_empty": sum("error" in c for c in calls),
        "input_Mtok": round(tin / 1e6, 2), "output_Mtok": round(tout / 1e6, 2),
        "usd_offpeak_no_cache": round(tin / 1e6 * pin + tout / 1e6 * pout, 2),
    }
    print(json.dumps(row, indent=1))
    return row


print("model:", MODEL, "| provider: deepseek | base_url: https://api.deepseek.com | temperature: 0.0 (fixed in llm.py)",
      "| max_tokens: 4096 | thinking:", os.environ.get("DEEPSEEK_THINKING", "disabled"),
      "| commit:", COMMIT, "| openai", version("openai"))


## 1. Smoke test: one call per model


In [ ]:
# One call per model through the repo's own wrapper, to check the key, the provider and that thinking is off
import openai
from dotenv import load_dotenv
load_dotenv(".env", override=True)
from utils import initialize_clients
from llm import timed_llm_call

client = initialize_clients("deepseek")[0]
for m in PRICES:
    text, info = timed_llm_call(client, "deepseek", m, "What is 15% of 300,000? Reply with the number only.",
                                role="generator", call_id=f"smoke_{m}", max_tokens=50)
    raw = client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say OK."}], max_tokens=5,
                                         extra_body={"thinking": {"type": "disabled"}})
    print(m, "| answer:", text.strip(), "| tokens:", info["prompt_num_tokens"], info["response_num_tokens"],
          "| served model:", raw.model, "| reasoning_content:", getattr(raw.choices[0].message, "reasoning_content", None))


## 2. Pilot: online ACE on the first 20 Formula questions (real calls, token counts and cost)


In [ ]:
# A copy of the repo whose Formula test split is cut to the first 20 questions; the main checkout stays untouched
!rm -rf /content/ace_pilot && rsync -a --exclude results /content/ace/ /content/ace_pilot/
lines = open("/content/ace_pilot/eval/finance/data/formula_test.jsonl").read().splitlines()[:20]
open("/content/ace_pilot/eval/finance/data/formula_test.jsonl", "w").write("\n".join(lines) + "\n")
run_ace("results/pilot_formula_online", cwd="/content/ace_pilot")
pilot = summarize("results/pilot_formula_online", cwd="/content/ace_pilot")
print(f"projected for 200 questions: about {pilot['calls'] * 10} calls, about ${pilot['usd_offpeak_no_cache'] * 10:.0f} off-peak")


## 3. Main run: online ACE on all 200 Formula questions, 3 runs


In [ ]:
# Online adaptation on the full Formula test split (200). Each run also reports the base LLM (empty playbook) first.
rows = []
for i in (1, 2, 3):
    path = f"results/formula_online_{MODEL}_r{MAX_ROUNDS}_run{i}"
    run_ace(path)
    rows.append(summarize(path))


In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nbase  mean %.3f  std %.3f" % (df.base_acc.mean(), df.base_acc.std()))
print("ACE   mean %.3f  std %.3f" % (df.ace_acc.mean(), df.ace_acc.std()))
print("paper (DeepSeek-V3.1, Table 2): base 0.675, ACE online 0.765")
